<a href="https://colab.research.google.com/github/technologymalviya/MachineLeaning/blob/main/Gita.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bhagavad Gita QA Model

Train a retrieval-based question-answering model on the **Srimad Bhagavad Gita** (700 verses) using Hugging Face datasets. The trained model is saved to `gita_model/` and can answer Gita-related queries.

In [15]:
# Install required packages (run once in Colab/local)
%pip install -q datasets scikit-learn joblib pandas numpy

In [16]:
# Colab: download gita_qa_model.py if not in the working directory
import os
import urllib.request

if not os.path.exists('gita_qa_model.py'):
    url = 'https://raw.githubusercontent.com/technologymalviya/MachineLeaning/main/gita_qa_model.py'
    urllib.request.urlretrieve(url, 'gita_qa_model.py')
    print('Downloaded gita_qa_model.py')
else:
    print('gita_qa_model.py already present')

Downloaded gita_qa_model.py


In [17]:
import json
import pandas as pd
from datasets import load_dataset
from gita_qa_model import GitaQAModel

## 1. Load Gita Datasets

- **Bhagavad-Gita_Dataset**: 700 verses in Sanskrit, Hindi, and English
- **Bhagavad-Gita-QA**: 3,500 verse-aligned question-answer pairs for training and evaluation

In [18]:
verses_ds = load_dataset("JDhruv14/Bhagavad-Gita_Dataset", split="train")
qa_ds = load_dataset("JDhruv14/Bhagavad-Gita-QA", split="train")

verses_df = verses_ds.to_pandas()
qa_df = qa_ds.to_pandas()

print(f"Verses: {len(verses_df)} | QA pairs: {len(qa_df)}")
print("\nSample verse:")
print(verses_df.iloc[0][['chapter', 'verse', 'english']])
print("\nSample QA:")
print(qa_df.iloc[0][['chapter_no', 'verse_no', 'question', 'answer']])

Verses: 701 | QA pairs: 3500

Sample verse:
chapter                                                    1
verse                                                      1
english    Dhritarashtra said: Sanjaya, gathered on the s...
Name: 0, dtype: object

Sample QA:
chapter_no                                                    1
verse_no                                                      1
question      Why does Dhritarashtra ask Sanjaya to describe...
answer        Dhritarashtra is blind, both physically and sy...
Name: 0, dtype: object


## 2. Train the Model

Hybrid retrieval model (`gita_qa_model.py`):
1. **TF-IDF verse retrieval** — finds the most relevant Gita verse(s) for a question
2. **QA matching** — picks the best answer from training Q&A pairs for the retrieved verse

In [19]:
model = GitaQAModel()
model.fit(verses_df, qa_df)
print(f"Training complete — {len(model.train_qa)} QA pairs used for training")

Training complete — 2800 QA pairs used for training


## 3. Test & Accuracy Report

Evaluation on a held-out **20% test set** (700 questions):

| Metric | Description |
|--------|-------------|
| **Verse Top-1 Accuracy** | Correct chapter.verse retrieved as #1 match |
| **Verse Top-3 Accuracy** | Correct verse found in top 3 results |
| **Answer Token F1** | Word overlap between predicted and expected answer |
| **Answer Cosine Similarity** | Semantic similarity of answers (TF-IDF) |
| **Overall Knowledge Score** | Weighted composite (40% verse + 60% answer) |

In [20]:
metrics = model.evaluate()

print("=" * 50)
print("  BHAGAVAD GITA MODEL — ACCURACY REPORT")
print("=" * 50)
for key, value in metrics.items():
    label = key.replace('_', ' ').title()
    suffix = '%' if 'accuracy' in key or 'f1' in key or 'similarity' in key or 'score' in key else ''
    print(f"  {label:<30} {value}{suffix}")
print("=" * 50)

  BHAGAVAD GITA MODEL — ACCURACY REPORT
  Test Samples                   700
  Train Samples                  2800
  Verse Top1 Accuracy            52.86%
  Verse Top3 Accuracy            67.0%
  Answer Token F1                25.03%
  Answer Cosine Similarity       13.71%
  Answer F1 When Verse Top3      26.07%
  Overall Knowledge Score        41.82%


## 4. Save Trained Model

The model is saved to `gita_model/gita_qa_model.joblib` for reuse in other scripts or applications.

In [21]:
model.save('gita_model')

Model saved to gita_model/


## 5. Try Gita Queries

Ask questions about the Bhagavad Gita — the model returns an answer with relevant verse references.

In [22]:
def print_gita_answer(result):
    print(f"Question: {result['question']}\n")
    print(f"Answer: {result['answer']}\n")
    print("Relevant Verses:")
    for v in result['verses']:
        print(f"  Chapter {v['chapter']}, Verse {v['verse']} (score: {v['score']:.3f})")
        print(f"    EN: {v['english'][:120]}...")
    print("-" * 60)

sample_questions = [
    "What does Krishna say about karma yoga?",
    "What is the nature of the soul according to the Gita?",
    "Why was Arjuna reluctant to fight?",
    "What is dharma?",
]

for q in sample_questions:
    print_gita_answer(model.ask(q))

Question: What does Krishna say about karma yoga?

Answer: Krishna first explains the attitude of mind to Arjuna through knowledge yoga, which focuses on understanding the true nature of the self and reality. He then introduces karma yoga, emphasizing the importance of performing one's duty without attachment. While knowledge yoga centers on wisdom and realization, karma yoga deals with disciplined action guided by that wisdom. Both paths aim to free one from the bondage of actions, but karma yoga is the practical application of wisdom in daily life.

Relevant Verses:
  Chapter 8, Verse 1 (score: 0.159)
    EN: Arjuna said: Krishna, what is that Brahma, what is Adhyatma, and what is Karma? What is called Adhibhutaand what is term...
  Chapter 2, Verse 39 (score: 0.147)
    EN: Arjuna, this attitude of mind has been declared to you in the knowledge yoga; now hear it in the karma yoga by which, be...
  Chapter 16, Verse 13 (score: 0.111)
    EN: They say to themselves, This much has been

## 6. Load Saved Model (for future use)

Use this cell in a new session to load the pre-trained model without retraining.

In [23]:
loaded_model = GitaQAModel.load('gita_model')

my_question = "What is the teaching on detachment?"
result = loaded_model.ask(my_question)
print_gita_answer(result)

Question: What is the teaching on detachment?

Answer: The 'most esoteric teaching' referred to in this verse is the knowledge imparted by Lord Krishna to Arjuna. It encapsulates profound spiritual wisdom that is essential for understanding the nature of the self and the universe.

Relevant Verses:
  Chapter 3, Verse 31 (score: 0.188)
    EN: Even those men who, with an uncavilling and devout mind, always follow this teaching of Mine are released from the bonda...
  Chapter 3, Verse 32 (score: 0.186)
    EN: They, however, who, finding fault with this teaching of Mine, do not follow it, take those fools to be deluded in the ma...
  Chapter 15, Verse 20 (score: 0.164)
    EN: Arjuna, this most esoteric teaching has thus been imparted by Me; grasping it in essence man becomes wise and his missio...
------------------------------------------------------------


In [26]:
sample_questions = [
     "What is dharma?"
]

for q in sample_questions:
    print_gita_answer(model.ask(q))

Question: What is dharma?

Answer: Krishna emphasizes faith in dharma because it is essential for spiritual progress and liberation. Faith acts as a guiding force, enabling individuals to transcend the cycle of birth and death and ultimately attain union with the divine.

Relevant Verses:
  Chapter 18, Verse 32 (score: 0.187)
    EN: The intellect which imagines even Adharma to be Dharma, and sees all other things upside-down,—wrapped in ignorance, tha...
  Chapter 9, Verse 3 (score: 0.176)
    EN: O Arjuna, those who have no faith in this dharma, O conqueror of enemies, they do not attain Me, but return to the cycle...
  Chapter 7, Verse 11 (score: 0.166)
    EN: O best of the Bharatas, I am the strength of the strong, devoid of desire and passion; and I am the desire in beings, no...
------------------------------------------------------------
